run from 716.2

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys
sys.path.append('/content/drive/MyDrive/dbms/spider code')

In [3]:
# =====================================================================
# SchemaFlow (prototype) - Text-to-SQL on Spider, for Google Colab
# Uses process_sql.py + evaluation.py from https://github.com/taoyds/spider
# Each "# %%" block = one Colab cell.  Runtime > Change runtime type > GPU
# =====================================================================

# %% [1] Install + get Spider repo (process_sql.py, evaluation.py)
# !pip -q install transformers sentencepiece nltk gdown
# !git clone https://github.com/taoyds/spider.git /content/spider

# %% [2] Get Spider DATA (database/, tables.json, train_spider.json, dev.json, dev_gold.sql)
# Download "Spider Dataset" from https://yale-lily.github.io/spider  (spider.zip)
# and either upload it to Colab or put it on Drive, then:
# !unzip -q /content/spider.zip -d /content/         # -> /content/spider/ (data folder)
# NOTE: the data zip also unpacks into a folder called "spider". To avoid mixing it with
# the git repo, rename after unzip:  !mv /content/spider /content/spider_data
# Expected: /content/spider_data/{database,tables.json,train_spider.json,dev.json,dev_gold.sql}

# %% [3] Config
import os, sys, json, math, random, sqlite3, subprocess, re
import nltk
nltk.download("punkt"); nltk.download("punkt_tab")
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import T5TokenizerFast, T5ForConditionalGeneration
from transformers.modeling_outputs import BaseModelOutput



[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [4]:
REPO   = "/content/drive/MyDrive/dbms/spider code"        # git clone (evaluation.py, process_sql.py)
DATA   = "/content/drive/MyDrive/dbms/spider_data"   # dataset
DB_DIR = f"{DATA}/database"
sys.path.insert(0, REPO)
from process_sql import get_schema, Schema, get_sql      # from taoyds/spider

# CFG.update(model_name="google/flan-t5-base", batch=4, lr=5e-5, epochs=8)

CFG = dict(
    model_name="google/flan-t5-base",   # t5-base gives better accuracy; needs more GPU memory
    max_src=512, max_tgt=200, batch=4, lr=5e-5, epochs=8,
    ode_steps=4,             # Euler steps for schema graph neural ODE
    lam_sch=0.0, lam_flow=0.01,
    train_limit=None,        # e.g. 2000 for a quick smoke test
    dev_limit=None,          # e.g. 200 for a quick smoke test
    beams=8,
)
dev = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(0); torch.manual_seed(0)

# %% [4] PROCESSING (uses process_sql.get_schema)
tokenizer = T5TokenizerFast.from_pretrained(CFG["model_name"])

def clean_sql_in(s):   # T5 vocab has no '<'
    return s.replace("<=", " LTE ").replace("<", " LT ")
def clean_sql_out(s):
    return s.replace(" LTE ", " <= ").replace(" LT ", " < ")

tables = {t["db_id"]: t for t in json.load(open(f"{DATA}/tables.json"))}

def build_schema_graph(db_id):
    """Nodes = tables + columns. Edges = table<->column, foreign keys (undirected), self loops."""
    t = tables[db_id]
    names, edges = [], []
    tab_idx = {}
    for i, tn in enumerate(t["table_names_original"]):
        tab_idx[i] = len(names); names.append(tn.lower())
    col_idx = {}
    for j, (ti, cn) in enumerate(t["column_names_original"]):
        if ti < 0: continue                       # skip '*'
        col_idx[j] = len(names); names.append(f"{t['table_names_original'][ti]} {cn}".lower())
        edges.append((col_idx[j], tab_idx[ti]))
    for a, b in t["foreign_keys"]:
        if a in col_idx and b in col_idx: edges.append((col_idx[a], col_idx[b]))
    for pk in t["primary_keys"]:
        pk = pk if isinstance(pk, int) else pk[0]
    return names, edges

def serialize(question, db_id):
    t = tables[db_id]; s = f"{question} | {db_id}"
    for i, tn in enumerate(t["table_names_original"]):
        cols = [c for (ti, c) in t["column_names_original"] if ti == i]
        s += f" | {tn.lower()} : " + ", ".join(c.lower() for c in cols)
    return s

class SpiderDS(Dataset):
    def __init__(self, path, limit=None, train=True):
        data = json.load(open(path))
        self.items = data[:limit] if limit else data
        self.train = train
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        ex = self.items[i]
        d = dict(src=serialize(ex["question"], ex["db_id"]), db_id=ex["db_id"])
        if self.train: d["tgt"] = clean_sql_in(ex["query"])
        return d

def collate(batch):
    enc = tokenizer([b["src"] for b in batch], max_length=CFG["max_src"], truncation=True,
                    padding=True, return_tensors="pt")
    out = dict(input_ids=enc.input_ids, attention_mask=enc.attention_mask)
    if "tgt" in batch[0]:
        lab = tokenizer([b["tgt"] for b in batch], max_length=CFG["max_tgt"], truncation=True,
                        padding=True, return_tensors="pt").input_ids
        lab[lab == tokenizer.pad_token_id] = -100
        out["labels"] = lab
    graphs = [build_schema_graph(b["db_id"]) for b in batch]
    N = max(len(g[0]) for g in graphs)
    adj = torch.zeros(len(batch), N, N); nmask = torch.zeros(len(batch), N, dtype=torch.bool)
    node_tok = torch.zeros(len(batch), N, 8, dtype=torch.long)     # up to 8 wordpieces / node
    node_tok_mask = torch.zeros(len(batch), N, 8)
    for b, (names, edges) in enumerate(graphs):
        nmask[b, :len(names)] = True
        for k, nm in enumerate(names):
            ids = tokenizer(nm, add_special_tokens=False).input_ids[:8]
            node_tok[b, k, :len(ids)] = torch.tensor(ids); node_tok_mask[b, k, :len(ids)] = 1
        for u, v in edges: adj[b, u, v] = adj[b, v, u] = 1
        adj[b, torch.arange(len(names)), torch.arange(len(names))] = 1
    out.update(adj=adj, node_mask=nmask, node_tok=node_tok, node_tok_mask=node_tok_mask,
               db_ids=[b["db_id"] for b in batch])
    return out



# %% [5] MODEL (REPLACES OLD CELL 5):  T5  +  Neural Sheaf Diffusion  +  Modern-Hopfield schema linking
# Everything else (cells 1-4, 6-9) stays exactly as you have it.
# Same class name / constructor / forward() / generate() interface -> training + eval code is untouched.
#
# New ideas (not used in the Spider/SPARC/CoSQL text-to-SQL papers):
#   1. Neural SHEAF diffusion on the schema graph  (Bodnar et al., NeurIPS 2022, arXiv:2202.04579)
#        - every schema edge (table-column, foreign key) learns its own linear "restriction maps",
#          so a column can be transformed differently when talking to its table vs. to a foreign key.
#   2. Modern HOPFIELD retrieval for schema linking (Ramsauer et al., ICLR 2021, arXiv:2008.02217)
#        - question tokens iteratively "retrieve" schema items from an associative memory (multi-step, learnable beta).
#   3. Zero-initialised residual fusion (ReZero-style): the model starts EXACTLY as pretrained T5 and
#      learns to inject schema information gradually (old LayerNorm(H+att) distorted T5's encoder output scale).
# Pure PyTorch, no extra installs.  Memory is O(B*N^2*k^2) with N = #schema nodes (<~150 in Spider) -> tiny.

class SheafDiffusion(nn.Module):
    """Dense sheaf diffusion.  Node state h_v in R^d is viewed as a stalk X_v in R^{k x c}  (d = k*c).
       Restriction maps  F_{u<e}  = I + 0.25*tanh(W_o tanh(W_a h_u + W_b h_v))  in R^{k x k}, for e=(u,v).
       Sheaf Laplacian   (L X)_u = 1/deg(u) * sum_v F_{u<e}^T (F_{u<e} X_u - F_{v<e} X_v)
       Update            X <- X - eta * L X ;  H <- H + FFN(LN(H))."""
    def __init__(self, d, k=4, r=32, steps=3):
        super().__init__()
        assert d % k == 0, "d_model must be divisible by stalk dim k"
        self.k, self.c, self.steps = k, d // k, steps
        self.pa, self.pb = nn.Linear(d, r), nn.Linear(d, r)
        self.to_map = nn.Linear(r, k * k)
        nn.init.zeros_(self.to_map.weight); nn.init.zeros_(self.to_map.bias)   # F = I at start (plain graph diffusion)
        self.log_step = nn.Parameter(torch.zeros(1))                            # eta = 0.25*sigmoid(.) <= 0.25 (stable)
        self.ffn = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 2 * d), nn.GELU(), nn.Linear(2 * d, d))
        nn.init.zeros_(self.ffn[-1].weight); nn.init.zeros_(self.ffn[-1].bias)
        self.register_buffer("eye", torch.eye(k))

    def forward(self, H, A):
        B, N, d = H.shape; k, c = self.k, self.c
        deg = A.sum(-1).clamp(min=1)[..., None, None]                           # B,N,1,1
        eta = 0.25 * torch.sigmoid(self.log_step)
        Aw = A[..., None, None]                                                 # B,N,N,1,1
        for _ in range(self.steps):
            z = torch.tanh(self.pa(H).unsqueeze(2) + self.pb(H).unsqueeze(1))   # B,N,N,r   (ordered pair u,v)
            M = self.eye + 0.25 * torch.tanh(self.to_map(z)).view(B, N, N, k, k)  # M[b,u,v] = F_{u<(u,v)}
            Mt = M.transpose(-1, -2)
            G = (Aw * (Mt @ M)).sum(2)                                          # B,N,k,k   sum_v F_u^T F_u
            T = Mt @ M.transpose(1, 2)                                          # B,N,N,k,k F_u^T F_v  (M[v,u] = F_{v<e})
            Tw = (Aw * T).permute(0, 1, 3, 2, 4).reshape(B, N * k, N * k)
            X = H.reshape(B, N, k, c)
            lap = ((G @ X) - (Tw @ X.reshape(B, N * k, c)).view(B, N, k, c)) / deg
            H = (X - eta * lap).reshape(B, N, d)
            H = H + self.ffn(H)
        return H

class HopfieldRetrieval(nn.Module):
    """Multi-head modern Hopfield layer: question tokens (queries) retrieve schema nodes (stored patterns).
       xi <- softmax(beta * xi K^T) K   (T-1 refinement steps),   R = softmax(beta * xi K^T) V."""
    def __init__(self, d, heads=8, steps=2):
        super().__init__()
        self.h, self.dh, self.steps = heads, d // heads, steps
        self.q, self.k, self.v = (nn.Linear(d, d, bias=False) for _ in range(3))
        self.o = nn.Linear(d, d)
        nn.init.zeros_(self.o.weight); nn.init.zeros_(self.o.bias)             # zero-init residual branch
        self.log_beta = nn.Parameter(torch.zeros(heads))

    def forward(self, Hq, S, smask):
        B, L, d = Hq.shape; N = S.size(1); h, dh = self.h, self.dh
        sp = lambda x, n: x.view(B, n, h, dh).transpose(1, 2)                   # B,h,n,dh
        xi, K, V = sp(self.q(Hq), L), sp(self.k(S), N), sp(self.v(S), N)
        beta = (self.log_beta.exp() / math.sqrt(dh)).view(1, h, 1, 1)
        neg = ~smask[:, None, None, :]
        for t in range(self.steps):
            att = torch.softmax((beta * xi @ K.transpose(-1, -2)).masked_fill(neg, -1e9), -1)
            if t < self.steps - 1: xi = att @ K                                 # Hopfield state update
        R = (att @ V).transpose(1, 2).reshape(B, L, d)
        return self.o(R)

# ---- unchanged helpers from your original code (normalizing-flow density head) ----
class AffineCoupling(nn.Module):
    def __init__(self, d):
        super().__init__(); self.d = d
        self.net = nn.Sequential(nn.Linear(d // 2, d), nn.ReLU(), nn.Linear(d, d))
    def forward(self, x):
        x1, x2 = x[..., :self.d // 2], x[..., self.d // 2:]
        s, t = self.net(x1).chunk(2, -1); s = torch.tanh(s)
        return torch.cat([x1, x2 * torch.exp(s) + t], -1), s.sum(-1)

class FlowHead(nn.Module):
    def __init__(self, d, K=2):
        super().__init__(); self.layers = nn.ModuleList([AffineCoupling(d) for _ in range(K)])
    def forward(self, x):
        logdet = 0
        for l in self.layers:
            x, ld = l(x); x = x.flip(-1); logdet = logdet + ld
        logp0 = -0.5 * (x ** 2).sum(-1) - 0.5 * x.size(-1) * math.log(2 * math.pi)
        return -(logp0 + logdet)

class SheafHopSQL(nn.Module):
    def __init__(self, name, ode_steps=3):          # ode_steps now = number of sheaf-diffusion steps (0 = ablation)
        super().__init__()
        self.t5 = T5ForConditionalGeneration.from_pretrained(name)
        d = self.t5.config.d_model
        self.node_proj = nn.Linear(d, d); self.node_ln = nn.LayerNorm(d)
        self.sheaf = SheafDiffusion(d, k=4, r=32, steps=ode_steps)
        self.q_ln = nn.LayerNorm(d)
        self.hop = HopfieldRetrieval(d, heads=8, steps=2)
        self.flow = FlowHead(d)

    def encode(self, b):
        H_tok = self.t5.encoder(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).last_hidden_state
        E = self.t5.shared(b["node_tok"]); m = b["node_tok_mask"].unsqueeze(-1)
        H0 = self.node_ln(self.node_proj((E * m).sum(2) / m.sum(2).clamp(min=1)))   # node init from wordpieces
        S = self.sheaf(H0, b["adj"])                                                # schema evolution (sheaf)
        R = self.hop(self.q_ln(H_tok), S, b["node_mask"])                           # schema linking (Hopfield)
        return H_tok + R                                                            # zero-init residual fusion

    def forward(self, b):
        fused = self.encode(b)
        out = self.t5(encoder_outputs=BaseModelOutput(last_hidden_state=fused),
                      attention_mask=b["attention_mask"], labels=b["labels"])
        flow_nll = self.flow(fused.detach())
        mask = b["attention_mask"].bool()
        flow_nll = flow_nll[mask].mean() / fused.size(-1)
        return out.loss + CFG["lam_flow"] * flow_nll, out.loss, flow_nll

    @torch.no_grad()
    def generate(self, b, beams=8):
        fused = self.encode(b)
        return self.t5.generate(encoder_outputs=BaseModelOutput(last_hidden_state=fused),
                                attention_mask=b["attention_mask"], num_beams=beams,
                                num_return_sequences=beams, max_length=CFG["max_tgt"])

SchemaFlow = SheafHopSQL     # alias -> cell 6 (`SchemaFlow(CFG["model_name"], CFG["ode_steps"])`) works unchanged





tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

In [5]:
# %% [6] TRAIN
def to_dev(b): return {k: (v.to(dev) if torch.is_tensor(v) else v) for k, v in b.items()}

model = SchemaFlow(CFG["model_name"], CFG["ode_steps"]).to(dev)
print("params (M):", sum(p.numel() for p in model.parameters()) / 1e6)

train_ds = SpiderDS(f"{DATA}/train_spider.json", CFG["train_limit"], train=True)
train_dl = DataLoader(train_ds, batch_size=CFG["batch"], shuffle=True, collate_fn=collate)
opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=0.01)
total = len(train_dl) * CFG["epochs"]
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / 200) * max(0.0, 1 - s / total))

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

params (M): 254.717017


In [6]:
model.train(); step = 0
for ep in range(CFG["epochs"]):
    for b in train_dl:
        loss, ce, fl = model(to_dev(b))
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step(); opt.zero_grad(); step += 1
        if step % 100 == 0: print(f"ep{ep} step{step}/{total} loss={loss.item():.3f} ce={ce.item():.3f} flow={fl.item():.3f}")
torch.save(model.state_dict(), "/content/drive/MyDrive/dbms/spider_data/modelstore/schemaflow.pt")




ep0 step100/14000 loss=1.049 ce=1.040 flow=0.902
ep0 step200/14000 loss=1.119 ce=1.116 flow=0.354
ep0 step300/14000 loss=0.691 ce=0.690 flow=0.112
ep0 step400/14000 loss=0.576 ce=0.575 flow=0.038
ep0 step500/14000 loss=0.404 ce=0.404 flow=0.026
ep0 step600/14000 loss=0.300 ce=0.300 flow=-0.002
ep0 step700/14000 loss=0.388 ce=0.388 flow=-0.006
ep0 step800/14000 loss=0.242 ce=0.242 flow=-0.017
ep0 step900/14000 loss=0.722 ce=0.722 flow=-0.018
ep0 step1000/14000 loss=0.470 ce=0.470 flow=-0.024
ep0 step1100/14000 loss=0.277 ce=0.277 flow=-0.027
ep0 step1200/14000 loss=0.436 ce=0.436 flow=-0.029
ep0 step1300/14000 loss=0.430 ce=0.430 flow=-0.031
ep0 step1400/14000 loss=0.338 ce=0.338 flow=-0.031
ep0 step1500/14000 loss=0.376 ce=0.377 flow=-0.035
ep0 step1600/14000 loss=0.154 ce=0.154 flow=-0.036
ep0 step1700/14000 loss=0.397 ce=0.397 flow=-0.033
ep1 step1800/14000 loss=0.312 ce=0.313 flow=-0.037
ep1 step1900/14000 loss=0.194 ce=0.195 flow=-0.035
ep1 step2000/14000 loss=0.146 ce=0.147 flow=-

In [7]:
# %% [7] SYMBOLIC VERIFICATION (grammar + schema check using Spider's process_sql + SQLite)
_schema_cache = {}
def is_valid(sql, db_id):
    """Hard constraints: (a) parses under Spider's SQL grammar with schema resolution,
       (b) SQLite can plan it (tables/columns exist)."""
    db = f"{DB_DIR}/{db_id}/{db_id}.sqlite"
    try:
        if db_id not in _schema_cache: _schema_cache[db_id] = Schema(get_schema(db))
        get_sql(_schema_cache[db_id], sql)                 # raises on invalid grammar / unknown names
        con = sqlite3.connect(db); con.execute("EXPLAIN " + sql); con.close()
        return True
    except Exception:
        return False

def verified_decode(cands, db_id):
    cands = [clean_sql_out(c).strip() for c in cands]
    for c in cands:                                        # best-scoring valid candidate
        if is_valid(c, db_id): return c, True
    return cands[0], False

In [8]:
# %% [8] PREDICT ON DEV
dev_ds = SpiderDS(f"{DATA}/dev.json", CFG["dev_limit"], train=False)
dev_dl = DataLoader(dev_ds, batch_size=4, shuffle=False, collate_fn=collate)
model.eval(); preds, n_valid = [], 0
for b in dev_dl:
    bd = to_dev(b)
    seqs = model.generate(bd, CFG["beams"])
    texts = tokenizer.batch_decode(seqs, skip_special_tokens=True)
    for i, db_id in enumerate(b["db_ids"]):
        sql, ok = verified_decode(texts[i * CFG["beams"]:(i + 1) * CFG["beams"]], db_id)
        preds.append(" ".join(sql.split())); n_valid += ok
print(f"valid SQL after verification: {n_valid}/{len(preds)}")
with open("/content/drive/MyDrive/dbms/spider_data/pred.txt", "w") as f: f.write("\n".join(preds) + "\n")



valid SQL after verification: 834/1034


In [9]:
# %% [9] EVALUATION (official evaluation.py, wrapped to tolerate non-UTF-8 text in Spider DBs)

gold_lines = open("/content/drive/MyDrive/dbms/spider_data/dev_gold.sql").read().strip().split("\n")[:len(preds)]
open("/content/drive/MyDrive/dbms/spider_data/gold.txt", "w").write("\n".join(gold_lines) + "\n")

wrapper = f'''
import sqlite3, sys, runpy, os
_orig = sqlite3.connect
def _connect(*a, **k):
    c = _orig(*a, **k)
    c.text_factory = lambda b: b.decode("utf-8", errors="ignore")
    return c
sqlite3.connect = _connect
sys.path.insert(0, {REPO!r})
sys.argv = ["evaluation.py"] + sys.argv[1:]
runpy.run_path({REPO + "/evaluation.py"!r}, run_name="__main__")
'''
open("/content/run_eval.py", "w").write(wrapper)

res = subprocess.run(
    ["python", "/content/run_eval.py",
     "--gold", "/content/drive/MyDrive/dbms/spider_data/gold.txt", "--pred", "/content/drive/MyDrive/dbms/spider_data/pred.txt",
     "--db", DB_DIR, "--table", "/content/drive/MyDrive/dbms/spider_data/tables.json", "--etype", "all"],
    capture_output=True, text=True)
print(res.stdout[-3000:]); print(res.stderr[-1000:])

    0.964                0.925                0.916                0.958               
IUEN                 0.000                0.000                0.656                0.429                0.519               
keywords             0.902                0.905                0.824                0.827                0.878               
---------------------- PARTIAL MATCHING RECALL ----------------------
select               0.831                0.762                0.713                0.596                0.744               
select(no AGG)       0.859                0.769                0.713                0.596                0.753               
where                0.778                0.632                0.415                0.266                0.550               
where(no OP)         0.806                0.637                0.468                0.362                0.588               
group(no Having)     0.800                0.774                0.872                0.

In [10]:
# =====================================================================
# Result-section builder: every number comes from the official taoyds/spider evaluation.py
# Paste after cell 9 (needs REPO, DATA, DB_DIR, subprocess, re). Works for the T5 and LLM versions.
# =====================================================================

# %% [11] Run + parse evaluation.py for ANY prediction file
import os, re, json, subprocess
import pandas as pd

LEVELS = ["easy", "medium", "hard", "extra", "all"]
PART = {"SELECT": "select", "WHERE": "where", "GROUP BY": "group", "ORDER BY": "order", "AND/OR": "and/or"}

WRAPPER = f'''
import sqlite3, sys, runpy
_orig = sqlite3.connect
def _connect(*a, **k):
    c = _orig(*a, **k)
    c.text_factory = lambda b: b.decode("utf-8", errors="ignore")
    return c
sqlite3.connect = _connect
sys.path.insert(0, {REPO!r})
sys.argv = ["evaluation.py"] + sys.argv[1:]
runpy.run_path({REPO + "/evaluation.py"!r}, run_name="__main__")
'''
open("/content/run_eval.py", "w").write(WRAPPER)

ROW = re.compile(r"^(select\(no AGG\)|select|where\(no OP\)|where|group\(no Having\)|group|order|and/or|IUEN|keywords)\s+([0-9\.\s]+)$")

def parse_all(stdout):
    """-> {'count','EX','EM','P','R','F1'} ; each metric maps level -> value in percent."""
    out = {"count": {}, "EX": {}, "EM": {}, "P": {}, "R": {}, "F1": {}}
    section = None
    for line in stdout.splitlines():
        s = line.strip(); up = s.upper()
        if "PARTIAL MATCHING ACCURACY" in up: section = "P"; continue
        if "PARTIAL MATCHING RECALL" in up:   section = "R"; continue
        if "PARTIAL MATCHING F1" in up:       section = "F1"; continue
        if "EXECUTION ACCURACY" in up or "EXACT MATCHING ACCURACY" in up: section = None; continue
        nums = lambda t: [float(x) for x in t.split()[-5:]]
        if s.startswith("count") and not out["count"]:
            out["count"] = dict(zip(LEVELS, [int(x) for x in s.split()[-5:]])); continue
        if s.startswith("execution") and not out["EX"]:
            out["EX"] = dict(zip(LEVELS, [round(v * 100, 1) for v in nums(s)])); continue
        if s.startswith("exact match") and not out["EM"]:
            out["EM"] = dict(zip(LEVELS, [round(v * 100, 1) for v in nums(s)])); continue
        m = ROW.match(s)
        if m and section:
            out[section][m.group(1)] = dict(zip(LEVELS, [round(float(x) * 100, 1) for x in m.group(2).split()[-5:]]))
    return out

def run_eval(pred_path, tag="run"):
    preds_ = open(pred_path, encoding="utf-8").read().rstrip("\n").split("\n")
    gold_ = open(f"{DATA}/dev_gold.sql", encoding="utf-8").read().strip().split("\n")[:len(preds_)]
    gpath = f"/content/gold_{tag}.txt"; open(gpath, "w", encoding="utf-8").write("\n".join(gold_) + "\n")
    r = subprocess.run(["python", "/content/run_eval.py", "--gold", gpath, "--pred", pred_path,
                        "--db", DB_DIR, "--table", f"{DATA}/tables.json", "--etype", "all"],
                       capture_output=True, text=True)
    parsed = parse_all(r.stdout)
    if not parsed["EX"]: raise RuntimeError("could not parse evaluation output:\n" + r.stdout[-1500:] + r.stderr[-1500:])
    return parsed

def flatten(p):
    """parsed run -> flat {metric: percent} using the 'all' column."""
    f = {"EM": p["EM"]["all"], "EX": p["EX"]["all"]}
    for name, key in PART.items():
        for sec in ["P", "R", "F1"]:
            f[f"{sec}:{name}"] = p[sec].get(key, {}).get("all")
    return f

# %% [12] Models in the table
# (a) YOUR model: evaluated from pred.txt produced by cell 8
ours = run_eval(f"{DATA}/pred.txt", "ours")
MODELS = {"SchemaFlow (ours)": flatten(ours)}

# (b) Baselines, honest option 1 (recommended): evaluate their released dev predictions with the SAME script.
#     Put the path of each model's predicted-SQL file (one SQL per line, Spider dev order) here.
BASELINE_PRED_FILES = {
    # "RASAT [11]":        "/content/drive/MyDrive/dbms/baselines/rasat_dev_pred.txt",
    # "LGESQL [8]":        "/content/drive/MyDrive/dbms/baselines/lgesql_dev_pred.txt",
    # "T5+PICARD [9]":     "/content/drive/MyDrive/dbms/baselines/picard_dev_pred.txt",
    # "DIN-SQL (GPT-4) [1]": "/content/drive/MyDrive/dbms/baselines/dinsql_dev_pred.txt",
}
for name, path in BASELINE_PRED_FILES.items():
    MODELS[name] = flatten(run_eval(path, re.sub(r"\W", "", name)))

# (c) Baselines, option 2: numbers copied from the papers. Keys: "EM","EX","P:SELECT","R:WHERE","F1:ORDER BY",...
#     Only enter what the paper REPORTS (same split + metric). Leave everything else as None -> printed as "--".
BASELINE_MANUAL = {
    # "RASAT [11]": {"EM": None, "EX": None},
}
for name, vals in BASELINE_MANUAL.items():
    MODELS[name] = vals

# %% [13] Build the table (same layout as your paper), print, save CSV + LaTeX
SECTIONS = [("Partial Matching Precision", "P"), ("Partial Matching Recall", "R"), ("Partial Matching F1", "F1")]
def get(m, k): return MODELS[m].get(k)

rows = [("Exact Match (EM)", "EM"), ("Execution Accuracy (EX)", "EX")]
for title, sec in SECTIONS:
    rows.append((title, None))
    rows += [(n, f"{sec}:{n}") for n in PART]

cols = list(MODELS)
fmt = lambda v: "--" if v is None else f"{v:.1f}%"
table = [[label] + ([""] * len(cols) if key is None else [fmt(get(m, key)) for m in cols]) for label, key in rows]
df = pd.DataFrame(table, columns=["Metric"] + cols)
print(df.to_string(index=False))
df.to_csv(f"{DATA}/results_table.csv", index=False)

def tex_row(label, key):
    if key is None: return f"\\multicolumn{{{len(cols)+1}}}{{l}}{{\\textit{{{label}}}}} \\\\\n"
    vals = [get(m, key) for m in cols]
    best = max([v for v in vals if v is not None], default=None)
    cells = [("--" if v is None else (f"\\textbf{{{v:.1f}\\%}}" if v == best else f"{v:.1f}\\%")) for v in vals]
    return label.replace("&", "\\&") + " & " + " & ".join(cells) + " \\\\\n"

tex = ("\\begin{table*}[t]\n\\centering\n\\caption{Comparison on the Spider development set (official evaluation script).}\n"
       "\\begin{tabular}{l" + "c" * len(cols) + "}\n\\toprule\nMetric & " + " & ".join(cols) + " \\\\\n\\midrule\n")
for i, (label, key) in enumerate(rows):
    if key is None and i > 0: tex += "\\midrule\n"
    tex += tex_row(label, key)
tex += "\\bottomrule\n\\end{tabular}\n\\end{table*}\n"

# %% [14] By-hardness table (EX and EM for every model that has a prediction file) + extras
hard = {"SchemaFlow (ours)": ours}
for name, path in BASELINE_PRED_FILES.items():
    hard[name] = run_eval(path, re.sub(r"\W", "", name))
tex += "\n\\begin{table}[t]\n\\centering\n\\caption{Execution accuracy by query hardness (Spider dev).}\n\\begin{tabular}{lccccc}\n\\toprule\n"
tex += "Model & Easy & Medium & Hard & Extra & All \\\\\n\\midrule\n"
for name, p in hard.items():
    tex += name + " & " + " & ".join(f"{p['EX'][l]:.1f}\\%" for l in LEVELS) + " \\\\\n"
tex += "\\bottomrule\n\\end{tabular}\n\\end{table}\n"
print("\nExecution accuracy by hardness:")
print(pd.DataFrame({n: p["EX"] for n, p in hard.items()}).T.to_string())
print("Question counts:", ours["count"])

open(f"{DATA}/results_tables.tex", "w", encoding="utf-8").write(tex)
print("\nsaved:", f"{DATA}/results_table.csv", f"{DATA}/results_tables.tex")
# LaTeX needs \usepackage{booktabs}.
# Also report: parameters (cell 6 print), valid SQL count (cell 8 print), and the error-analysis table
# from cell 12 of schemaflow_results.py (works unchanged with this notebook).

                    Metric SchemaFlow (ours)
          Exact Match (EM)             56.4%
   Execution Accuracy (EX)             56.0%
Partial Matching Precision                  
                    SELECT             91.9%
                     WHERE             77.4%
                  GROUP BY             76.7%
                  ORDER BY             81.1%
                    AND/OR             95.8%
   Partial Matching Recall                  
                    SELECT             74.4%
                     WHERE             55.0%
                  GROUP BY             73.1%
                  ORDER BY             72.3%
                    AND/OR             99.7%
       Partial Matching F1                  
                    SELECT             82.2%
                     WHERE             64.3%
                  GROUP BY             74.9%
                  ORDER BY             76.4%
                    AND/OR             97.7%

Execution accuracy by hardness:
                   eas